In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

staging_orders = spark.table("brazilian_ecommerce.prod.staging_orders")
staging_order_items = spark.table("brazilian_ecommerce.prod.staging_order_items")
staging_order_payments = spark.table("brazilian_ecommerce.prod.staging_order_payments")

# --- aggregate order items: total items, total price, total freight ---
order_items_agg = (
    staging_order_items
    .groupBy("order_id")
    .agg(
        F.count("order_item_id").alias("total_items"),
        F.sum("price").alias("total_price"),
        F.sum("freight_value").alias("total_freight_value"),
    )
)

# --- aggregate payments: primary (most common) payment type + max installments + total paid ---
payment_type_counts = (
    staging_order_payments
    .groupBy("order_id", "payment_type")
    .agg(F.count("*").alias("type_count"))
)

payment_type_window = Window.partitionBy("order_id").orderBy(F.desc("type_count"))

primary_payment_type = (
    payment_type_counts
    .withColumn("rank", F.row_number().over(payment_type_window))
    .filter(F.col("rank") == 1)
    .select("order_id", F.col("payment_type").alias("primary_payment_type"))
)

payments_agg = (
    staging_order_payments
    .groupBy("order_id")
    .agg(
        F.max("payment_installments").alias("payment_installments"),
        F.sum("payment_value").alias("total_paid_value"),
    )
    .join(primary_payment_type, on="order_id", how="left")
)

# --- build the gold sales summary ---
gold_sales_summary = (
    staging_orders
    .join(order_items_agg, on="order_id", how="left")
    .join(payments_agg, on="order_id", how="left")
    .withColumn("total_order_value", F.col("total_price") + F.col("total_freight_value"))
    .select(
        "order_id",
        "customer_id",
        "order_status",
        "order_purchase_timestamp",
        "total_items",
        "total_price",
        "total_freight_value",
        "total_order_value",
        "primary_payment_type",
        "payment_installments",
        "total_paid_value",
        "delivery_status",
        "hours_purchase_to_approved",
        "hours_approved_to_carrier",
        "hours_carrier_to_customer",
        "hours_estimated_to_delivered",
    )
)

full_table_name = "brazilian_ecommerce.prod.gold_sales_summary"

(
    gold_sales_summary.write
    .mode("overwrite")
    .option("comment", "gold table: one row per order, with item/payment aggregates and delivery metrics")
    .option("overwriteSchema", "true")
    .saveAsTable(full_table_name)
)

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ALTER COLUMN order_id SET NOT NULL
""")

spark.sql(f"""
    ALTER TABLE {full_table_name}
    ADD CONSTRAINT pk_gold_sales_summary PRIMARY KEY (order_id)
""")

print(f"Loaded {gold_sales_summary.count()} rows into {full_table_name}")